# TAPNext++ Torch Demo

This Colab provides a demo of TAPNext++, a state-of-the-art model for online point tracking in videos.
It demonstrates:
* How to load a pretrained TAPNext++ model.
* How to run online inference to track points frame-by-frame.
* How to evaluate tracking performance on the TAP-Vid-DAVIS dataset using standard metrics like Average Jaccard (AJ), Occlusion Accuracy (OA) and Average Point-to-Point Similarity (PTS).

Only 'first frame' query modes for TAP-Vid-DAVIS is demonstrated as 'strided' is not supported by TAPNext++.

### Download model



In [1]:
!pwd

/data/pbk5339/thesis/DiffTrack/point_track/tapnextpp_test


In [2]:
!wget --no-check-certificate https://storage.googleapis.com/dm-tapnet/tapnextpp/tapnextpp_ckpt.pt

--2026-04-15 09:27:10--  https://storage.googleapis.com/dm-tapnet/tapnextpp/tapnextpp_ckpt.pt
Resolving storage.googleapis.com (storage.googleapis.com)... 142.251.167.207, 172.253.122.207, 172.253.63.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|142.251.167.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2532282370 (2.4G) [application/octet-stream]
Saving to: ‘tapnextpp_ckpt.pt’

tapnextpp_ckpt.pt   100%[===================>]   2.36G   232MB/s    in 11s     

2026-04-15 09:27:21 (224 MB/s) - ‘tapnextpp_ckpt.pt’ saved [2532282370/2532282370]



### Download dataset

In [3]:
!wget --no-check-certificate https://storage.googleapis.com/dm-tapnet/tapvid_davis.zip
!unzip tapvid_davis.zip

--2026-04-15 09:27:29--  https://storage.googleapis.com/dm-tapnet/tapvid_davis.zip
Resolving storage.googleapis.com (storage.googleapis.com)... 64.233.180.207, 142.251.163.207, 142.251.16.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|64.233.180.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1668710491 (1.6G) [application/zip]
Saving to: ‘tapvid_davis.zip’

tapvid_davis.zip    100%[===================>]   1.55G   120MB/s    in 16s     

2026-04-15 09:27:45 (102 MB/s) - ‘tapvid_davis.zip’ saved [1668710491/1668710491]

Archive:  tapvid_davis.zip
   creating: tapvid_davis/
  inflating: tapvid_davis/SOURCES.md  
  inflating: tapvid_davis/README.md  
  inflating: tapvid_davis/tapvid_davis.pkl  


In [4]:
import torch
import torchvision

In [5]:
torch.__version__, torchvision.__version__

('2.2.2+cu118', '0.17.2+cu118')

In [6]:
!pip install -q git+https://github.com/google-deepmind/tapnet.git

In [7]:
!pip install -q git+https://github.com/google-deepmind/recurrentgemma.git@main

ERROR: Package 'recurrentgemma' requires a different Python: 3.10.19 not in '<4.0,>=3.11'


In [8]:
!pip install "numpy<2.1.0"

In [9]:
import tqdm

In [10]:
from tapnet import evaluation_datasets

davis_dataset = evaluation_datasets.create_davis_dataset(
    davis_points_path='tapvid_davis/tapvid_davis.pkl',
    query_mode='first',
    full_resolution=False,
    resolution=(256, 256),
)

cached_dataset = []
for j, batch in enumerate(davis_dataset):
  cached_dataset.append(batch)
  print(
      'video id',
      j,
  )

I0000 00:00:1776259988.256413 1058742 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1776260017.768187 1058742 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1776260017.770054 1058742 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


KeyboardInterrupt: 

### TAPNext++

In [16]:
import numpy as np
from tapnet.tapnext.tapnext_torch import TAPNext
from tapnet.tapnext.tapnext_torch_utils import tracker_certainty
import torch.nn.functional as F

/scratch/pbk5339/conda_envs/difftrack/lib/python3.10/site-packages/torch/_export/utils.py:125: UserWarning: torch.utils._pytree._register_pytree_node is deprecated. Please use torch.utils._pytree.register_pytree_node instead.
  _register_pytree_node(


In [ ]:
def run_eval_per_frame(
    model,
    batch,
    get_trackwise_metrics=True,
    radius=8,
    threshold=0.5,
    use_certainty=False,
):
  with torch.no_grad():
    pred_tracks, track_logits, visible_logits, tracking_state = model(
        video=batch['video'][:, :1], query_points=batch['query_points']
    )
    pred_visible = visible_logits > 0
    pred_tracks, pred_visible = [pred_tracks.cpu()], [pred_visible.cpu()]
    pred_track_logits, pred_visible_logits = [track_logits.cpu()], [
        visible_logits.cpu()
    ]
    for frame in tqdm.tqdm(range(1, batch['video'].shape[1])):
      # ***************************************************
      # HERE WE RUN POINT TRACKING IN PURELY ONLINE FASHION
      # ***************************************************
      (
          curr_tracks,
          curr_track_logits,
          curr_visible_logits,
          tracking_state,
      ) = model(
          video=batch['video'][:, frame : frame + 1],
          state=tracking_state,
      )
      curr_visible = curr_visible_logits > 0
      # ***************************************************
      pred_tracks.append(curr_tracks.cpu())
      pred_visible.append(curr_visible.cpu())
      pred_track_logits.append(curr_track_logits.cpu())
      pred_visible_logits.append(curr_visible_logits.cpu())
    tracks = torch.cat(pred_tracks, dim=1).transpose(1, 2)
    pred_visible = torch.cat(pred_visible, dim=1).transpose(1, 2)
    track_logits = torch.cat(pred_track_logits, dim=1).transpose(1, 2)
    visible_logits = torch.cat(pred_visible_logits, dim=1).transpose(1, 2)

    pred_certainty = tracker_certainty(tracks, track_logits, radius)
    pred_visible_and_certain = (
        F.sigmoid(visible_logits) * pred_certainty
    ) > threshold

    if use_certainty:
      occluded = ~(pred_visible_and_certain.squeeze(-1))
    else:
      occluded = ~(pred_visible.squeeze(-1))

  scalars = evaluation_datasets.compute_tapvid_metrics(
      batch['query_points'].cpu().numpy(),
      batch['occluded'].cpu().numpy(),
      batch['target_points'].cpu().numpy(),
      occluded.numpy() + 0.0,
      tracks.numpy()[..., ::-1],
      query_mode='first',
      get_trackwise_metrics=get_trackwise_metrics,
  )
  return (
      tracks.numpy()[..., ::-1],
      occluded.numpy(),
      {k: v.sum(0) for k, v in scalars.items()},
  )


# @title Function for raw data to the input format {form-width: "25%"}
def deterministic_eval(cached_dataset, strided=False):
  if not strided:
    for sample in cached_dataset:
      batch = sample['davis'].copy()
      # batch['video'] = (batch['video'] + 1) / 2
      batch['visible'] = np.logical_not(batch['occluded'])[..., None]
      batch['padding'] = np.ones(
          batch['query_points'].shape[:2], dtype=np.bool_
      )
      batch['loss_mask'] = np.ones(
          batch['target_points'].shape[:3] + (1,), dtype=np.float32
      )
      batch['appearance'] = np.ones(
          batch['target_points'].shape[:3] + (1,), dtype=np.float32
      )

      yield batch
  else:
    for sample in cached_dataset:
      batch = sample['davis'].copy()
      # batch['video'] = (batch['video'] + 1) / 2
      batch['visible'] = np.logical_not(batch['occluded'])[..., None]
      batch['padding'] = np.ones(
          batch['query_points'].shape[:2], dtype=np.bool_
      )
      batch['loss_mask'] = np.ones(
          batch['target_points'].shape[:3] + (1,), dtype=np.float32
      )
      batch['appearance'] = np.ones(
          batch['target_points'].shape[:3] + (1,), dtype=np.float32
      )
      backward_batch = {k: v.copy() for k, v in batch.items()}
      for key in ['visible', 'appearance', 'loss_mask', 'target_points']:
        backward_batch[key] = np.flip(backward_batch[key], axis=2)
      backward_batch['video'] = np.flip(backward_batch['video'], axis=1)
      backward_queries = (
          backward_batch['video'].shape[1]
          - backward_batch['query_points'][..., 0]
          - 1
      )
      backward_batch['query_points'][..., 0] = backward_queries
      yield batch, backward_batch

### Create the model and load checkpoint

In [ ]:
model = TAPNext(image_size=(256, 256))
ckpt_path = 'tapnextpp_ckpt.pt'
ckpt = torch.load(ckpt_path, map_location='cpu')
model.load_state_dict({k.replace('tapnext.', ''): v for k, v in ckpt['state_dict'].items()})
model.cuda()

### Run inference

In [ ]:
standard_eval_scalars_list = []
preds = []
for batch in deterministic_eval(cached_dataset):
  batch = {k: torch.from_numpy(v).cuda().float() for k, v in batch.items()}
  with torch.amp.autocast('cuda', dtype=torch.float16, enabled=True):
    tracks, occluded, scores = run_eval_per_frame(
        model, batch, get_trackwise_metrics=False, use_certainty=False
    )
  standard_eval_scalars_list.append(scores)
  preds.append((tracks, occluded))


print('')
print(
    'AJ',
    np.mean([
        standard_eval_scalars_list[k]['average_jaccard']
        for k in range(len(standard_eval_scalars_list))
    ]),
)
print(
    'OA',
    np.mean([
        standard_eval_scalars_list[k]['occlusion_accuracy']
        for k in range(len(standard_eval_scalars_list))
    ]),
)
print(
    'PTS',
    np.mean([
        standard_eval_scalars_list[k]['average_pts_within_thresh']
        for k in range(len(standard_eval_scalars_list))
    ]),
)

In [ ]:
davis_dataset_strided = evaluation_datasets.create_davis_dataset(
    davis_points_path='tapvid_davis/tapvid_davis.pkl',
    query_mode='strided',
    full_resolution=False,
    resolution=(256, 256),
)

cached_dataset_strided = []
for j, batch in enumerate(davis_dataset_strided):
  cached_dataset_strided.append(batch)
  print('video id', j)

In [ ]:
import jax

eval_results = list()
for vid, (fbatch, bbatch) in enumerate(deterministic_eval(cached_dataset_strided, strided=True)):
  fbatch = {k: torch.from_numpy(v).cuda().float() for k, v in fbatch.items()}
  bbatch = {k: torch.from_numpy(v.copy()).cuda().float() for k, v in bbatch.items()}
  with torch.amp.autocast('cuda', dtype=torch.float16, enabled=True):
    ftracks, foccluded, _ = run_eval_per_frame(model, fbatch, get_trackwise_metrics=False, use_certainty=False)
    btracks, boccluded, _ = run_eval_per_frame(model, bbatch, get_trackwise_metrics=False, use_certainty=False)
  btracks, boccluded = np.flip(btracks, axis=2), np.flip(boccluded, axis=2)
  # tracks = [1, q, t, 2]
  for q in range(fbatch['query_points'].shape[1]):
    t = int((fbatch['query_points'][0, q, 0]).item())
    ftracks[0, q, :t] = btracks[0, q, :t]
    foccluded[0, q, :t] = boccluded[0, q, :t]
  tracks, occluded = ftracks, foccluded
  scalars = evaluation_datasets.compute_tapvid_metrics(
      cached_dataset_strided[vid]['davis']['query_points'],
      cached_dataset_strided[vid]['davis']['occluded'],
      cached_dataset_strided[vid]['davis']['target_points'],
      occluded + 0.,
      tracks,
      query_mode='strided',
      get_trackwise_metrics=False,
  )
  eval_results.append(jax.tree.map(lambda x: np.array(np.sum(x, axis=0)), scalars))

print('')
print(
    'AJ',
    np.mean([
        eval_results[k]['average_jaccard']
        for k in range(len(eval_results))
    ]),
)
print(
    'OA',
    np.mean([
        eval_results[k]['occlusion_accuracy']
        for k in range(len(eval_results))
    ]),
)
print(
    'PTS',
    np.mean([
        eval_results[k]['average_pts_within_thresh']
        for k in range(len(eval_results))
    ]),
)

In [20]:
import imageio
import cv2
import numpy as np
from IPython.display import HTML, display
import base64
import matplotlib.pyplot as plt


In [19]:
import imageio
import cv2
import numpy as np
from IPython.display import HTML, display
import base64
import matplotlib.pyplot as plt

def visualize_davis_tracks(video, tracks, occluded, output_gif_path="output.gif", fps=10):
    """
    Creates a GIF from the tracking results and visualizes it in the notebook.

    Args:
      video: Array of shape (T, H, W, 3) containing the video frames.
      tracks: Array of shape (N, T, 2) containing point coordinates (x, y).
      occluded: Array of shape (N, T) containing boolean occlusion masks.
      output_gif_path: Local path to save the GIF.
      fps: Frames per second for the GIF.
    """
    in_memory_frames = []
    N, T, _ = tracks.shape

    # Generate distinct colors for each point
    colors = plt.cm.hsv(np.linspace(0, 1, N))[:, :3] * 255

    for t in range(T):
        frame = video[t]
        # Handle different frame value scales (e.g., [-1, 1], [0, 1], or [0, 255])
        if frame.dtype in [np.float32, np.float64]:
            if frame.min() < 0:
                frame = ((frame + 1) / 2 * 255).astype(np.uint8)
            else:
                frame = (frame * 255).astype(np.uint8)
        else:
            frame = frame.copy()

        # Draw visible points
        for n in range(N):
            if not occluded[n, t]:
                x, y = int(tracks[n, t, 0]), int(tracks[n, t, 1])
                # Ensure points are within frame bounds
                if 0 <= x < frame.shape[1] and 0 <= y < frame.shape[0]:
                    cv2.circle(frame, (x, y), 2, colors[10].tolist(), -1)

        in_memory_frames.append(frame)

    # Save GIF locally
    imageio.mimsave(output_gif_path, in_memory_frames, fps=fps, loop=0)
    print(f"GIF saved to {output_gif_path}")

    # Read and display in notebook
    with open(output_gif_path, "rb") as f:
        gif_bytes = f.read()
    gif_base64 = base64.b64encode(gif_bytes).decode("ascii")
    display(HTML(f'<img src="data:image/gif;base64,{gif_base64}" width="30%"/>'))

# Visualize the first video from the dataset using the first prediction
vid_idx = 10
demo_video = cached_dataset[vid_idx]['davis']['video'][0]  # (T, H, W, 3)
demo_tracks = preds[vid_idx][0][0]  # (N, T, 2)
demo_occluded = preds[vid_idx][1][0]  # (N, T)

visualize_davis_tracks(demo_video, demo_tracks, demo_occluded)

NameError: name 'cached_dataset' is not defined

### Run on a Custom Video
Here, we strip away the evaluation metrics from `run_eval_per_frame` (which requires ground-truth labels) to create a `run_inference_custom_video` function. This matches the exact step-by-step causal tracking logic used for the benchmark, but applies it to your own `.mp4` video with uniformly sampled query points at $t=0$.

In [26]:
import torchvision.io
import torchvision.transforms.functional as TF
import torch.nn.functional as F
from tapnet.tapnext.tapnext_torch_utils import tracker_certainty

def run_inference_custom_video(
    model,
    video_tensor,
    query_points,
    radius=8,
    threshold=0.5,
):
    """
    Runs TAPNext++ purely forward on a custom unlabelled video.
    video_tensor: (1, T, H, W, 3) between [-1, 1], float32
    """
    with torch.no_grad():
        with torch.amp.autocast('cuda', dtype=torch.float16, enabled=True):
            pred_tracks, track_logits, visible_logits, tracking_state = model(
                video=video_tensor[:, :1], query_points=query_points
            )
            pred_visible = visible_logits > 0

            pred_tracks, pred_visible = [pred_tracks.cpu()], [pred_visible.cpu()]
            pred_track_logits, pred_visible_logits = [track_logits.cpu()], [visible_logits.cpu()]

            for frame in tqdm.tqdm(range(1, video_tensor.shape[1])):
                (
                    curr_tracks,
                    curr_track_logits,
                    curr_visible_logits,
                    tracking_state,
                ) = model(
                    video=video_tensor[:, frame : frame + 1],
                    state=tracking_state,
                )
                curr_visible = curr_visible_logits > 0

                pred_tracks.append(curr_tracks.cpu())
                pred_visible.append(curr_visible.cpu())
                pred_track_logits.append(curr_track_logits.cpu())
                pred_visible_logits.append(curr_visible_logits.cpu())

            tracks = torch.cat(pred_tracks, dim=1).transpose(1, 2)
            track_logits = torch.cat(pred_track_logits, dim=1).transpose(1, 2)
            visible_logits = torch.cat(pred_visible_logits, dim=1).transpose(1, 2)

            pred_certainty = tracker_certainty(tracks, track_logits, radius)
            pred_visible_and_certain = (
                torch.sigmoid(visible_logits) * pred_certainty
            ) > threshold

            occluded = ~(pred_visible_and_certain.squeeze(-1))

    return tracks.numpy()[..., ::-1], occluded.numpy()

def format_custom_video(video_path, resolution=(256, 256), grid_size=16):
    """
    Loads MP4, resizes it, and maps query points over the first frame.
    """
    video, _, _ = torchvision.io.read_video(video_path, pts_unit="sec")

    video_resized = video.permute(0, 3, 1, 2)
    video_resized = torch.stack([TF.resize(frame, resolution) for frame in video_resized])
    video_resized = video_resized.permute(0, 2, 3, 1)

    # Output pure [0, 255] uint8 array so plotting functions don't improperly scale it
    vis_video = np.ascontiguousarray(video_resized.numpy().astype(np.uint8))
    
    video_tensor = (video_resized.float() / 255.0) * 2.0 - 1.0
    video_tensor = video_tensor.unsqueeze(0).cuda()

    points = []
    for y in np.linspace(0, resolution[0] - 1, grid_size):
        for x in np.linspace(0, resolution[1] - 1, grid_size):
            points.append([0, y, x])

    query_points = torch.tensor(points).unsqueeze(0).cuda().float()

    return video_tensor, query_points, vis_video

def visualize_custom_tracks(video, tracks, occluded, output_mp4_path="output.mp4", fps=24):
    """
    Creates an MP4 directly from imageio, mimicking the original GIF script completely natively.
    It loops colors so the points are rainbow and not hardcoded to Orange (Google's bug: colors[10]).
    """
    in_memory_frames = []
    N, T, _ = tracks.shape

    # Generate distinct colors for each point
    colors = plt.cm.hsv(np.linspace(0, 1, N))[:, :3] * 255

    for t in range(T):
        # We enforce uint8 natively from format_custom_video now, so just copy the frame
        frame = video[t].copy()
        frame = np.ascontiguousarray(frame)

        # Draw visible points
        for n in range(N):
            if not occluded[n, t]:
                x, y = int(tracks[n, t, 0]), int(tracks[n, t, 1])
                # Ensure points are within frame bounds
                if 0 <= x < frame.shape[1] and 0 <= y < frame.shape[0]:
                    cv2.circle(frame, (x, y), 2, tuple(int(c) for c in colors[n]), -1)

        in_memory_frames.append(frame)

    # Save MP4 locally (using imageio exactly like the gif saver, just different extension)
    imageio.mimsave(output_mp4_path, in_memory_frames, fps=fps, macro_block_size=None)
    print(f"MP4 saved to {output_mp4_path}")

    # Display dynamically in HTML player
    with open(output_mp4_path, "rb") as f:
        video_bytes = f.read()
    video_base64 = base64.b64encode(video_bytes).decode("ascii")
    display(HTML(f'''
    <video width="80%" controls autoplay loop>
        <source src="data:video/mp4;base64,{video_base64}" type="video/mp4">
    </video>
    '''))

In [ ]:
import numpy as np


In [ ]:
# my_video_path = "/scratch/pbk5339/thesis/DiffTrack/vids_mp4/swim.mp4"  
my_video_path = "/scratch/pbk5339/thesis/DiffTrack/UCF_Rep/val/v_BreastStroke_g24_c01.mp4"
# 2. Format the video into tensors and generate the initial tracking grid points
print(f"Processing video: {my_video_path}")
video_tensor, query_points, vis_video = format_custom_video(
    my_video_path, 
    resolution=(256, 256), 
    grid_size=15
)

# 3. Run the TAPNext++ causal tracking
# The "threshold" parameter controls when points vanish. 
# It checks: (visibility_confidence * position_certainty) > threshold.
# Default is 0.5. Lower it to make points stay visible longer, or set to 0.0 to never hide them.
print("Running tracking inference...")
tracks, occluded = run_inference_custom_video(
    model, 
    video_tensor, 
    query_points,
    threshold=0.15  # <-- Lowered threshold so points don't vanish as easily
)

# 4. Generate the tracked video natively using ImageIO like the GIF original
print("Generating visualization...")
visualize_custom_tracks(
    vis_video, 
    tracks[0], 
    occluded[0], 
    output_mp4_path="custom_tracked.mp4", 
    fps=24
)

Processing video: /scratch/pbk5339/thesis/DiffTrack/UCF_Rep/val/v_BreastStroke_g24_c01.mp4
Running tracking inference...
Running tracking inference...


100%|██████████| 116/116 [00:02<00:00, 38.67it/s]



Generating visualization...
MP4 saved to custom_tracked.mp4
MP4 saved to custom_tracked.mp4
